In [ ]:
!pip install -q langchain langchain-openai langchain-community chromadb

In [ ]:
import os
from google.colab import userdata

os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')
os.environ["GITHUB_BASE_URL"] = userdata.get('GITHUB_BASE_URL')



In [ ]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
llm = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o-mini",
    temperature=0.3
)

In [ ]:
#prueba de conexion con modelo
respuesta = llm.invoke("Hola")

print(respuesta.content)

In [ ]:
documentos = [
    Document(
        page_content="La asistencia mínima requerida para aprobar una asignatura es 70%."
    ),
    Document(
        page_content="La matrícula debe realizarse dentro de las fechas establecidas por la institución."
    ),
    Document(
        page_content="El retiro de asignaturas puede solicitarse dentro del plazo académico correspondiente."
    )
]

In [ ]:
embeddings = OpenAIEmbeddings(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="text-embedding-3-small"
)


In [ ]:
!pip install -q faiss-cpu

In [ ]:
db = FAISS.from_documents(
    documentos,
    embeddings
)

In [ ]:
resultados = db.similarity_search(
    "¿Cuál es la asistencia mínima para aprobar?"
)

print(resultados[0].page_content)

In [ ]:
pregunta = "¿Cual es la asistencia mínima para aprobar?"

contexto = db.similarity_search(
    pregunta,
    k=1
)[0].page_content

prompt = f"""
  Responde ultilizando solamente la siguiente información:

  {contexto}

  pregunta: {pregunta}
"""

respuesta = llm.invoke(prompt)
print(respuesta.content)

In [ ]:
from langchain_core.tools import tool

@tool
def buscar_reglamento(pregunta: str):
    """
    Busca información académica en documentos institucionales.
    """

    resultado = db.similarity_search(
        pregunta,
        k=1
    )

    return resultado[0].page_content

In [ ]:
print(
    buscar_reglamento.invoke(
        "¿Cuál es la asistencia mínima para aprobar?"
    )
)

In [ ]:
historial = []

def guardar_memoria(pregunta, respuesta):
    historial.append({
        "pregunta": pregunta,
        "respuesta": respuesta
    })

In [ ]:
guardar_memoria(
    "¿Cuál es la asistencia mínima?",
    "La asistencia mínima es 70%"
)

print(historial)

In [ ]:
def consultar_agente(pregunta):
    respuesta = buscar_reglamento.invoke(pregunta)
    guardar_memoria(
        pregunta,
        respuesta
    )
    return respuesta

In [ ]:
print(
    consultar_agente(
        "¿Cuál es la asistencia mínima para aprobar?"
    )
)

In [ ]:
print(historial)

In [ ]:
def agente_decisor(pregunta):

    if "asistencia" in pregunta.lower():
        return consultar_agente(pregunta)

    return "No encontré información relacionada en el reglamento."

In [ ]:
print(
    agente_decisor(
        "¿Cuál es la asistencia mínima?"
    )
)

In [ ]:
print(
    agente_decisor(
        "¿Cuál es el clima hoy?"
    )
)